In [43]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
from torch import nn as nn
from tqdm import tqdm

model = nn.Sequential(nn.Flatten(),
                      nn.Linear(784, 100),
                      nn.ReLU(),
                      nn.Linear(100, 100),
                      nn.ReLU(),
                      nn.Linear(100, 10))

opt = torch.optim.AdamW(model.parameters(), lr=0.0001)
ema_avg = lambda averaged_model_parameter, model_parameter, num_averaged: 0.95 * averaged_model_parameter + 0.05 * model_parameter
ema_model = torch.optim.swa_utils.AveragedModel(model)

transform = transforms.Compose(
    [transforms.ToTensor()]
    )


In [44]:
# Create datasets for training & validation, download if necessary
training_set = torchvision.datasets.MNIST('./data', train=True, transform=transform, download=True)
validation_set = torchvision.datasets.MNIST('./data', train=False, transform=transform, download=True)

# Create data loaders for our datasets; shuffle for training, not for validation
training_loader = torch.utils.data.DataLoader(training_set, batch_size=256, shuffle=True)
validation_loader = torch.utils.data.DataLoader(validation_set, batch_size=256, shuffle=False)

In [45]:
for i in range(1000):
    temp_loss = .0
    temp_ema_loss = .0
    temp_size = .0
    temp_ema_acc = .0
    for (x,y) in training_loader:
        output = model(x)
        loss = torch.nn.CrossEntropyLoss()(output, y)
        loss.backward()
        # Adjust learning weights
        opt.step()
        ema_model.update_parameters(model)
        y_pred = ema_model(x)
        ema_loss = torch.nn.CrossEntropyLoss()(y_pred,y).item()
        temp_loss += loss.item()
        temp_ema_loss += ema_loss
        temp_size += x.shape[0]

        temp_ema_acc += (y_pred.argmax(-1) == y).sum()
    print(f"{i} epoch passed TRAIN: the loss is {temp_loss/temp_size}, ema loss is {temp_ema_loss/temp_size}, acc {temp_ema_acc/temp_size}")

    with torch.no_grad():
        temp_loss_v = .0
        temp_ema_loss_v = .0
        temp_size_v = .0
        temp_ema_acc = .0
        temp_mod_acc = .0

        for (x,y) in validation_loader:
            output = model(x)
            loss = torch.nn.CrossEntropyLoss()(output, y)
            y_pred = ema_model(x)
            ema_loss = torch.nn.CrossEntropyLoss()(y_pred,y).item()
            temp_loss_v += loss.item()
            temp_ema_loss_v += ema_loss
            temp_size_v += x.shape[0]

            temp_ema_acc += (y_pred.argmax(-1) == y).sum()
            temp_mod_acc+= (model(x).argmax(-1) == y).sum()

        print(f"{i} epoch passed VAL: the loss is {temp_loss_v/temp_size_v}, ema loss is {temp_ema_loss_v/temp_size_v}, ema_acc {temp_ema_acc/temp_size_v}, acc_model {temp_mod_acc/temp_size_v}")
### Conclusion EMA is a must!!!

0 epoch passed TRAIN: the loss is 0.005657224984467029, ema loss is 0.007565791936715444, acc 0.5941833257675171
0 epoch passed VAL: the loss is 0.0024191656917333603, ema loss is 0.005344056797027588, ema_acc 0.7576000094413757, acc_model 0.8173999786376953
1 epoch passed TRAIN: the loss is 0.0034962714249889056, ema loss is 0.0034077716062466306, acc 0.8121500015258789
1 epoch passed VAL: the loss is 0.005440303033590317, ema loss is 0.002264027890563011, ema_acc 0.847100019454956, acc_model 0.7437999844551086
2 epoch passed TRAIN: the loss is 0.003412539127965768, ema loss is 0.0019584775537252424, acc 0.8565000295639038
2 epoch passed VAL: the loss is 0.002610622075200081, ema loss is 0.0017191846862435341, ema_acc 0.8758000135421753, acc_model 0.7820000052452087
3 epoch passed TRAIN: the loss is 0.0024206218202908834, ema loss is 0.0018228687355915706, acc 0.8631500005722046
3 epoch passed VAL: the loss is 0.0020857960924506188, ema loss is 0.00183236393481493, ema_acc 0.865499973

KeyboardInterrupt: 